In [1]:
!pip install groq chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    F

In [2]:
import os
os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"

In [3]:
import chromadb
from groq import Groq

# Step 1: Set up Vector DB with some documents
client_db = chromadb.Client()
collection = client_db.get_or_create_collection("rag_demo")

documents = [
    "The company refund policy allows returns within 30 days of purchase.",
    "To get a refund, customers must provide their order number and reason for return.",
    "Refunds are processed within 5-7 business days after approval.",
    "Products must be in original condition to qualify for a refund.",
    "Digital products and gift cards are not eligible for refunds."
]

collection.add(
    documents=documents,
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"]
)

# Step 2: User asks a question
user_question = "Are digital products eligible for refunds?"

# Step 3: Search Vector DB for relevant documents
results = collection.query(
    query_texts=[user_question],
    n_results=2
)

retrieved_docs = results['documents'][0]
context = "\n".join(retrieved_docs)

print("Retrieved from Vector DB:")
print(context)
print("\n" + "="*50 + "\n")

# Step 4: Send to LLM with context
llm_client = Groq()

response = llm_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Answer the question based ONLY on the provided context. If the answer is not in the context, say 'I don't know'."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {user_question}"}
    ]
)

print("LLM Answer:")
print(response.choices[0].message.content)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:00<00:00, 103MiB/s]


Retrieved from Vector DB:
Digital products and gift cards are not eligible for refunds.
Products must be in original condition to qualify for a refund.


LLM Answer:
Digital products and gift cards are not eligible for refunds.
